# Sales & Profit Intelligence

**Portfolio case study · Synthetic data · Python / Pandas / Matplotlib**

This notebook calculates sales and profit metrics, checks data quality, compares product/category/region performance, and visualizes monthly trends. The dataset is synthetic and does not describe a real company.

## 1. Imports and data loading

Run from this notebook's location. The CSV path is relative to the notebook.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path('../data/sales_data.csv')
df = pd.read_csv(DATA_PATH, parse_dates=['order_date'])
df.head()

## 2. Initial data-quality checks

Check shape, column types, missing values, duplicate rows, and invalid numeric values before calculating metrics.

In [ ]:
print(f'Rows: {len(df):,} | Columns: {len(df.columns)}')
display(df.dtypes.to_frame('dtype'))
display(df.isna().sum().to_frame('missing_values'))
print('Duplicate rows:', int(df.duplicated().sum()))

required_numeric = ['units_sold', 'unit_price', 'unit_cost', 'discount_rate']
invalid = (df[required_numeric].lt(0).any(axis=1) | df['discount_rate'].gt(1))
print('Rows with invalid numeric values:', int(invalid.sum()))
assert not df['order_date'].isna().any(), 'Missing order dates found'
assert not invalid.any(), 'Invalid numeric values found; investigate before analysis'

## 3. Calculate business metrics

Revenue reflects discounts. Profit is revenue less product cost. Profit margin is profit divided by revenue; it is not markup on cost.

In [ ]:
df['gross_sales'] = df['units_sold'] * df['unit_price']
df['revenue'] = df['gross_sales'] * (1 - df['discount_rate'])
df['cost'] = df['units_sold'] * df['unit_cost']
df['profit'] = df['revenue'] - df['cost']
df['profit_margin'] = df['profit'].div(df['revenue'].where(df['revenue'].ne(0)))
df['month'] = df['order_date'].dt.to_period('M').astype(str)

summary = pd.DataFrame({
    'metric': ['Orders/rows', 'Units sold', 'Revenue', 'Cost', 'Profit', 'Profit margin'],
    'value': [len(df), df['units_sold'].sum(), df['revenue'].sum(), df['cost'].sum(), df['profit'].sum(), df['profit'].sum() / df['revenue'].sum()]
})
display(summary)

## 4. Product, category, and region performance

In [ ]:
def performance_table(group_col):
    result = df.groupby(group_col, dropna=False).agg(
        orders=('order_date', 'size'),
        units=('units_sold', 'sum'),
        revenue=('revenue', 'sum'),
        cost=('cost', 'sum'),
        profit=('profit', 'sum'),
    )
    result['profit_margin'] = result['profit'] / result['revenue'].where(result['revenue'].ne(0))
    return result.sort_values('profit', ascending=False)

display(performance_table('product').round(2))
display(performance_table('category').round(2))
display(performance_table('region').round(2))

## 5. Visualize the results

These charts use calculated values from the dataset. Labels and chart titles state what is being measured.

In [ ]:
monthly = df.groupby('month', as_index=False).agg(revenue=('revenue', 'sum'), profit=('profit', 'sum'))
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(monthly['month'], monthly['revenue'], marker='o', label='Revenue')
ax.plot(monthly['month'], monthly['profit'], marker='o', label='Profit')
ax.set_title('Monthly Revenue and Profit (Synthetic Data)')
ax.set_xlabel('Month')
ax.set_ylabel('Amount (dataset currency units)')
ax.tick_params(axis='x', rotation=45)
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
by_product = performance_table('product').sort_values('profit', ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(by_product.index, by_product['profit'])
ax.set_title('Profit by Product (Synthetic Data)')
ax.set_xlabel('Profit (dataset currency units)')
ax.set_ylabel('Product')
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
plt.show()

## 6. Findings to investigate

The following cell prints the strongest observed groups. Treat these as findings from a synthetic demonstration dataset only, not as claims about a real business. Review the tables and charts before writing recommendations.

In [ ]:
category_results = performance_table('category')
region_results = performance_table('region')
product_results = performance_table('product')
print('Highest-profit category:', category_results['profit'].idxmax())
print('Highest-profit region:', region_results['profit'].idxmax())
print('Highest-profit product:', product_results['profit'].idxmax())
print(f"Overall profit margin: {df['profit'].sum() / df['revenue'].sum():.1%}")

print('\nBusiness questions to investigate:')
print('- Do high-revenue products also have strong profit margins?')
print('- Which regions have opportunities to improve revenue or profitability?')
print('- How do discounts affect profit across products?')
print('- Are monthly changes large enough to merit further investigation?')

## 7. Next steps

- Add tests for metric calculations and data validation.
- Compare discount scenarios without claiming causal effects.
- Recreate the core KPIs in Power BI.
- Replace the synthetic data only with a properly licensed public dataset or authorized business data.
- Update the report with verified results after running the notebook.